In [0]:
#Reading all the Bronze tables and fixing them to their individual dataframes
customers_df = spark.read.table("projects.project_1.bronze_customers")
order_item_df = spark.read.table("projects.project_1.bronze_order_items")
order_payments_df = spark.read.table("projects.project_1.bronze_order_payments")
orders_df = spark.read.table("projects.project_1.bronze_orders")
products_df = spark.read.table("projects.project_1.bronze_products")
product_category_df = spark.read.table("projects.project_1.bronze_products_category")
sellers_df = spark.read.table("projects.project_1.bronze_sellers")
geo_df = spark.read.table("projects.project_1.geo_location_bronze") 
order_reviews_df = spark.read.table("projects.project_1.order_reviews_bronze")

In [0]:
from pyspark.sql.functions import *
customers_df = customers_df.withColumn("customer_city", initcap("customer_city"))

In [0]:
bronze_orders_df = orders_df.join(order_item_df, on = "order_id", how = "inner").join(order_payments_df, on = "order_id", how ="inner")
bronze_orders_df = bronze_orders_df.select("order_id", "customer_id", "order_status", "order_purchase_timestamp","order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date", "order_item_id","product_id","seller_id","shipping_limit_date","price","freight_value","payment_sequential","payment_type","payment_installments","payment_value")

bronze_orders_df = bronze_orders_df.withColumn("order_purchase_timestamp", to_date("order_purchase_timestamp").alias("purchase_date"))\
                      .withColumn("order_approved_at", to_date("order_approved_at"))\
                      .withColumn("order_delivered_carrier_date", to_date("order_delivered_carrier_date"))\
                      .withColumn("order_delivered_customer_date", to_date("order_delivered_customer_date"))\
                      .withColumn("order_estimated_delivery_date", to_date("order_estimated_delivery_date"))\
                      .withColumn("shipping_limit_date", to_date("shipping_limit_date"))
display(bronze_orders_df)

In [0]:
from pyspark.sql.functions import *
#df = spark.read.table("projects.project_1.bronze_customers")
#display(df)
#Identified the Orders which still need to be delivered and their respective estimated delivery date
df_1 = spark.read.table("projects.project_1.bronze_orders")
df_pending = df_1.filter("order_status = 'invoiced'")
date_df = df_pending.withColumn("purchase_date", to_date("order_purchase_timestamp"))\
                    .withColumn("estimated_delivery_date", to_date("order_estimated_delivery_date"))

clean_df = date_df.select("order_id","order_status","purchase_date","estimated_delivery_date")
display(clean_df.distinct())

In [0]:
order_item_df = spark.read.table("projects.project_1.bronze_order_items")
product_df = spark.read.table("projects.project_1.bronze_products")
product_df = product_df.dropna(subset=["product_category_name"])
#display(product_df)
joined_df = clean_df.join(order_item_df,on = "order_id", how = "inner")
joined_2_df = joined_df.join(product_df, on = "product_id", how = "inner")
final_df = joined_2_df.select("order_status","purchase_date","estimated_delivery_date","product_category_name")
#display(final_df)
agg_df = final_df.groupBy("product_category_name").count().orderBy("count", ascending=False)
display(agg_df)